In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Map Initialization
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

# DEM collection, ~30m resolution
dataset = (
    ee.ImageCollection('COPERNICUS/DEM/GLO30')
    .filterBounds(panama_geom)
)

# Create a single DEM image and clip it
elevation = (
    dataset
    .select('DEM')
    .first()
    .clip(panama_geom)
)

# ~30 m resolution
slope = ee.Image("projects/deforestation-495419/assets/panama_slopee").clip(panama_geom)

# reprojecting to fit other data layers
sample_image = elevation
collection_projection = sample_image.projection()

reprojected_slope = (
    slope.resample("bilinear")
    .reproject(collection_projection)
    .rename("slope_reprojected")
    .clip(panama_geom)
)

Map.addLayer(slope, {"min": 0, "max": 30}, "Slope")
Map.addLayer(reprojected_slope, {"min": 0, "max": 30}, "Reprojected Slope")

Map 

Map(center=[8.515838945899919, -80.10966640141515], controls=(WidgetControl(options=['position', 'transparent_…